# Azure Document Intelligence 파싱 테스트

`pdfplumber`/`pypdf`로는 표 구조(셀 병합 등)가 깨지는 PDF를 Azure Document Intelligence(`prebuilt-layout`)로 다시 파싱해보기 위한 테스트 노트북입니다. FastAPI 앱과는 별개로 이 노트북에서만 동작합니다.

## 사전 준비
1. Azure Portal에서 **Document Intelligence** 리소스를 생성하고, 키/엔드포인트를 확인합니다.
2. 프로젝트 루트(`azure-doc-ai-service/`)의 `.env` 파일에 아래 값을 채웁니다 (없다면 `.env.example`을 복사).

```
AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT=https://<your-resource-name>.cognitiveservices.azure.com/
AZURE_DOCUMENT_INTELLIGENCE_API_KEY=<your-api-key>
```

3. 커널 실행 전, 아래 패키지가 설치되어 있어야 합니다.

```bash
.\.venv\Scripts\python.exe -m pip install jupyter ipykernel azure-ai-documentintelligence python-dotenv
```

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

# 이 노트북(azure-doc-ai-service/notebooks/)의 부모 폴더(azure-doc-ai-service/)에 있는 .env를 로드
ENV_PATH = Path.cwd().parent / ".env"
load_dotenv(dotenv_path=ENV_PATH)

DI_ENDPOINT = os.environ["AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT"]
DI_API_KEY = os.environ["AZURE_DOCUMENT_INTELLIGENCE_API_KEY"]

SAMPLES_DIR = Path.cwd().parent / "data" / "sample"
OUT_DIR = Path.cwd().parent / "data" / "parsed" / "document_intelligence"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("ENDPOINT:", DI_ENDPOINT)
print("SAMPLES_DIR:", SAMPLES_DIR)

In [ ]:
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.core.credentials import AzureKeyCredential

client = DocumentIntelligenceClient(endpoint=DI_ENDPOINT, credential=AzureKeyCredential(DI_API_KEY))

## 1. 단일 파일 테스트 (표가 깨졌던 파일로 검증)

`pdfplumber`가 셀 병합을 제대로 못 잡았던 `위치기반서비스+이용약관(별표)` 파일로 먼저 테스트합니다. `prebuilt-layout` 모델은 표/구조 인식용 모델입니다.

In [ ]:
TEST_FILENAME = "위치기반서비스+이용약관(별표)_20260331_V5.8.pdf"

test_path = SAMPLES_DIR / TEST_FILENAME

with open(test_path, "rb") as f:
    content = f.read()

print("file_size_bytes:", len(content))
print("first_4_bytes:", content[:4])  # PDF는 b'%PDF'로 시작해야 정상

poller = client.begin_analyze_document(
    "prebuilt-layout", body=content, content_type="application/octet-stream"
)

result = poller.result()  # 제출 -> 폴링 -> 완료까지 블로킹 대기

print("pages:", len(result.pages) if result.pages else 0)
print("tables:", len(result.tables) if result.tables else 0)
print("content_chars:", len(result.content) if result.content else 0)
print("--- content preview (앞 500자) ---")
print((result.content or "")[:500])

## 2. 표(table) 구조 확인

`result.tables`의 각 표는 `cells` 리스트를 갖고, 각 셀에 `row_index`/`column_index`가 있어 실제 행/열로 복원할 수 있습니다.

In [ ]:
def table_to_grid(table) -> list[list[str]]:
    grid = [["" for _ in range(table.column_count)] for _ in range(table.row_count)]
    for cell in table.cells:
        grid[cell.row_index][cell.column_index] = cell.content
    return grid


for i, table in enumerate(result.tables or []):
    print(f"=== table {i}: {table.row_count} rows x {table.column_count} cols ===")
    for row in table_to_grid(table):
        print(row)
    print()

## 3. 마크다운 형식 출력 (LLM 투입용, 선택)

`output_content_format="markdown"`으로 요청하면 표가 마크다운 표 형식으로 포함된 `result.content`를 받을 수 있습니다. SDK 버전에 따라 파라미터명이 다를 수 있으니 에러 시 설치된 `azure-ai-documentintelligence` 버전의 시그니처를 확인하세요.

In [ ]:
from azure.ai.documentintelligence.models import DocumentContentFormat

md_poller = client.begin_analyze_document(
    "prebuilt-layout",
    body=content,
    content_type="application/octet-stream",
    output_content_format=DocumentContentFormat.MARKDOWN,
)

md_result = md_poller.result()
print((md_result.content or "")[:1500])

## 4. 파일 1개당 전용 폴더에 4종 결과 저장 (기본 파싱 결과 / LLM용 마크다운 / 메타데이터 / 원본 PDF)

파일 하나를 분석해서, **그 파일 이름으로 된 전용 폴더**에 아래 내용을 저장하는 헬퍼 함수입니다.

```
data/parsed/document_intelligence/
└── <파일명(확장자 제외)>/
    ├── raw.json        # 기본 파싱 결과 (원시 구조화 데이터: pages, tables, bounding box, confidence 등)
    ├── content.md       # LLM 투입용 마크다운 (표가 마크다운 표로 변환되어 있음)
    ├── metadata.json     # 이 파일의 메타데이터 (페이지 수, 표 개수, 글자 수, 처리 시간 등)
    └── <원본파일명>.pdf   # 원본 PDF 복사본 (원본 samples 폴더와 별개로 결과 폴더에도 보관)
```


In [ ]:
import json
import shutil
import time

from azure.ai.documentintelligence.models import DocumentContentFormat


def run_di(filename: str, model_id: str = "prebuilt-layout") -> dict:
    path = SAMPLES_DIR / filename
    file_out_dir = OUT_DIR / path.stem
    file_out_dir.mkdir(parents=True, exist_ok=True)

    with open(path, "rb") as f:
        file_bytes = f.read()

    t0 = time.perf_counter()
    poller = client.begin_analyze_document(
        model_id,
        body=file_bytes,
        content_type="application/octet-stream",
        output_content_format=DocumentContentFormat.MARKDOWN,
    )
    result = poller.result()
    elapsed = round(time.perf_counter() - t0, 2)

    # 1) 기본 파싱 결과 (원시 구조화 데이터, JSON 전체 저장)
    raw_json_path = file_out_dir / "raw.json"
    raw_json_path.write_text(
        json.dumps(result.as_dict(), ensure_ascii=False, indent=2), encoding="utf-8"
    )

    # 2) LLM 투입용 마크다운
    md_path = file_out_dir / "content.md"
    md_path.write_text(result.content or "", encoding="utf-8")

    # 3) 원본 PDF 복사본 (같은 폴더에 원본 그대로 보관)
    original_pdf_path = file_out_dir / path.name
    shutil.copy2(path, original_pdf_path)

    # 4) 메타데이터
    meta = {
        "filename": filename,
        "size_bytes": path.stat().st_size,
        "pages_analyzed": len(result.pages) if result.pages else 0,
        "tables_detected": len(result.tables) if result.tables else 0,
        "content_chars": len(result.content or ""),
        "elapsed_sec": elapsed,
        "raw_json_file": str(raw_json_path),
        "markdown_file": str(md_path),
        "original_pdf_file": str(original_pdf_path),
        "error": None,
    }
    meta_path = file_out_dir / "metadata.json"
    meta_path.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
    meta["metadata_file"] = str(meta_path)

    return meta


# 표 구조가 중요한 파일 하나로 먼저 테스트 (전체를 다 돌리기 전 동작 확인용)
TEST_FILENAMES = [
    "위치기반서비스+이용약관(별표)_20260331_V5.8.pdf",
]

for name in TEST_FILENAMES:
    print(run_di(name))

In [ ]:
pdf_files = sorted(SAMPLES_DIR.glob("*.pdf"))
print(f"총 {len(pdf_files)}개 파일 처리 시작")

metadata_list = []
for i, path in enumerate(pdf_files, start=1):
    print(f"[{i}/{len(pdf_files)}] {path.name} 처리 중...")
    try:
        meta = run_di(path.name)
        print(
            f"  -> pages={meta['pages_analyzed']} tables={meta['tables_detected']} "
            f"chars={meta['content_chars']} ({meta['elapsed_sec']}s)"
        )
    except Exception as exc:  # noqa: BLE001
        error_msg = f"{type(exc).__name__}: {exc}"
        meta = {
            "filename": path.name,
            "size_bytes": path.stat().st_size,
            "pages_analyzed": None,
            "tables_detected": None,
            "content_chars": None,
            "elapsed_sec": None,
            "raw_json_file": None,
            "markdown_file": None,
            "error": error_msg,
        }
        # 실패한 파일도 전용 폴더에 에러 메타데이터를 남김
        file_out_dir = OUT_DIR / path.stem
        file_out_dir.mkdir(parents=True, exist_ok=True)
        meta_path = file_out_dir / "metadata.json"
        meta_path.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
        meta["metadata_file"] = str(meta_path)
        print(f"  -> 실패: {error_msg}")

    metadata_list.append(meta)
    time.sleep(1)  # 무료 티어 분당 호출 제한 대비 짧은 대기

# 전체 요약 (파일별 폴더의 metadata.json들을 한 곳에 모은 것)
metadata_path = OUT_DIR / "_metadata.json"
metadata_path.write_text(json.dumps(metadata_list, ensure_ascii=False, indent=2), encoding="utf-8")
print("\n전체 완료. 파일별 폴더:", OUT_DIR)
print("전체 요약 메타데이터:", metadata_path)

n_errors = sum(1 for m in metadata_list if m.get("error"))
print(f"성공: {len(metadata_list) - n_errors} / 실패: {n_errors}")